In [1]:
# This notebook establishes exactly what the AMEX files contain
# before we make any analytical or modeling decisions.

from pathlib import Path
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt


PROJECT_ROOT = Path.cwd()

if PROJECT_ROOT.name == "notebooks":
    PROJECT_ROOT = PROJECT_ROOT.parent


RAW_DIR = (
    PROJECT_ROOT
    / "data"
    / "raw"
)


TRAIN_PATH = (
    RAW_DIR
    / "train_data.csv"
)


LABEL_PATH = (
    RAW_DIR
    / "train_labels.csv"
)


print(
    "train_data exists:",
    TRAIN_PATH.exists(),
)

print(
    "train_labels exists:",
    LABEL_PATH.exists(),
)

train_data exists: True
train_labels exists: True


In [2]:
# train_labels is small enough to read completely.
# This gives us exact customer count and exact default rate.

labels = pd.read_csv(
    LABEL_PATH,
    dtype={
        "customer_ID": "string",
        "target": "int8",
    },
)


print(
    "Label rows:",
    f"{len(labels):,}",
)

print(
    "Unique customers:",
    f"{labels['customer_ID'].nunique():,}",
)

print(
    "Duplicate customers:",
    labels[
        "customer_ID"
    ].duplicated().sum(),
)

print(
    "Default rate:",
    f"{labels['target'].mean():.4%}",
)


labels.head()

Label rows: 458,913


Unique customers: 458,913
Duplicate customers: 0
Default rate: 25.8934%


,customer_ID,target
0,0000099d6bd597052cdcda90ffabf56573fe9d7c79be5f...,0
1,00000fd6641609c6ece5454664794f0340ad84dddce9a2...,0
2,00001b22f846c82c51f6e3958ccd81970162bae8b007e8...,0
3,000041bdba6ecadd89a52d11886e8eaaec9325906c9723...,0
4,00007889e4fcd2614b6cbe7f8f3d2e5c728eca32d9eb8a...,0


In [3]:
# We use a manageable sample for schema exploration.
# Loading the complete statement file into pandas would consume
# unnecessary memory for questions that only require column structure.

sample = pd.read_csv(
    TRAIN_PATH,
    nrows=250_000,
)


print(
    "Sample rows:",
    f"{len(sample):,}",
)

print(
    "Total columns:",
    len(sample.columns),
)


sample.head()

Sample rows: 250,000
Total columns: 190


,customer_ID,S_2,P_2,D_39,B_1,B_2,R_1,S_3,D_41,B_3,...,D_136,D_137,D_138,D_139,D_140,D_141,D_142,D_143,D_144,D_145
0,0000099d6bd597052cdcda90ffabf56573fe9d7c79be5f...,2017-03-09,0.938469,0.001733,0.008724,1.006838,0.009228,0.124035,0.008771,0.004709,...,NaN,NaN,NaN,0.002427,0.003706,0.003818,NaN,0.000569,0.000610,0.002674
1,0000099d6bd597052cdcda90ffabf56573fe9d7c79be5f...,2017-04-07,0.936665,0.005775,0.004923,1.000653,0.006151,0.126750,0.000798,0.002714,...,NaN,NaN,NaN,0.003954,0.003167,0.005032,NaN,0.009576,0.005492,0.009217
2,0000099d6bd597052cdcda90ffabf56573fe9d7c79be5f...,2017-05-28,0.954180,0.091505,0.021655,1.009672,0.006815,0.123977,0.007598,0.009423,...,NaN,NaN,NaN,0.003269,0.007329,0.000427,NaN,0.003429,0.006986,0.002603
3,0000099d6bd597052cdcda90ffabf56573fe9d7c79be5f...,2017-06-13,0.960384,0.002455,0.013683,1.002700,0.001373,0.117169,0.000685,0.005531,...,NaN,NaN,NaN,0.006117,0.004516,0.003200,NaN,0.008419,0.006527,0.009600
4,0000099d6bd597052cdcda90ffabf56573fe9d7c79be5f...,2017-07-16,0.947248,0.002483,0.015193,1.000727,0.007605,0.117325,0.004653,0.009312,...,NaN,NaN,NaN,0.003671,0.004946,0.008889,NaN,0.001670,0.008126,0.009827


In [4]:
# Repeated customer IDs confirm that the grain is
# one customer-statement snapshot rather than one customer.

sample_counts = (
    sample[
        "customer_ID"
    ]
    .value_counts()
)


print(
    sample_counts.describe()
)


example_customer = (
    sample_counts.index[0]
)


sample.loc[
    sample[
        "customer_ID"
    ] == example_customer,
    [
        "customer_ID",
        "S_2",
    ],
].sort_values(
    "S_2"
)

count    20736.000000
mean        12.056327
std          2.591282
min          1.000000
25%         13.000000
50%         13.000000
75%         13.000000
max         13.000000
Name: count, dtype: float64


,customer_ID,S_2
0,0000099d6bd597052cdcda90ffabf56573fe9d7c79be5f...,2017-03-09
1,0000099d6bd597052cdcda90ffabf56573fe9d7c79be5f...,2017-04-07
2,0000099d6bd597052cdcda90ffabf56573fe9d7c79be5f...,2017-05-28
3,0000099d6bd597052cdcda90ffabf56573fe9d7c79be5f...,2017-06-13
4,0000099d6bd597052cdcda90ffabf56573fe9d7c79be5f...,2017-07-16
5,0000099d6bd597052cdcda90ffabf56573fe9d7c79be5f...,2017-08-04
6,0000099d6bd597052cdcda90ffabf56573fe9d7c79be5f...,2017-09-18
7,0000099d6bd597052cdcda90ffabf56573fe9d7c79be5f...,2017-10-08
8,0000099d6bd597052cdcda90ffabf56573fe9d7c79be5f...,2017-11-20
9,0000099d6bd597052cdcda90ffabf56573fe9d7c79be5f...,2017-12-04


In [5]:
# S_2 is the statement date.
# Dates must be parsed before chronological operations.

sample[
    "S_2"
] = pd.to_datetime(
    sample[
        "S_2"
    ],
    errors="coerce",
)


print(
    "Sample date range:",
    sample["S_2"].min(),
    "to",
    sample["S_2"].max(),
)

Sample date range: 2017-03-01 00:00:00 to 2018-03-31 00:00:00


In [6]:
# customer_ID and S_2 are metadata.
# The remaining columns are anonymized behavioral variables.

metadata_columns = [
    "customer_ID",
    "S_2",
]


feature_columns = [
    column
    for column in sample.columns
    if column not in metadata_columns
]


families = {
    "D":
        [
            c for c
            in feature_columns
            if c.startswith("D_")
        ],

    "S":
        [
            c for c
            in feature_columns
            if c.startswith("S_")
        ],

    "P":
        [
            c for c
            in feature_columns
            if c.startswith("P_")
        ],

    "B":
        [
            c for c
            in feature_columns
            if c.startswith("B_")
        ],

    "R":
        [
            c for c
            in feature_columns
            if c.startswith("R_")
        ],
}


family_counts = pd.DataFrame(
    [
        {
            "family": family,
            "features": len(columns),
        }
        for family, columns
        in families.items()
    ]
)


print(
    "Behavioral feature count:",
    len(feature_columns),
)


family_counts

Behavioral feature count: 188


,family,features
0,D,96
1,S,21
2,P,3
3,B,40
4,R,28


In [7]:
# These AMEX variables are categorical even when
# their stored values look numeric.

categorical_features = [
    "B_30",
    "B_38",
    "D_114",
    "D_116",
    "D_117",
    "D_120",
    "D_126",
    "D_63",
    "D_64",
    "D_66",
    "D_68",
]


print(
    "Known categorical features:",
    len(categorical_features),
)

Known categorical features: 11


In [8]:
# Missingness is calculated by feature and then summarized by family.
# This tells us whether certain behavioral groups are substantially
# less complete than others.

missing_by_feature = (
    sample[
        feature_columns
    ]
    .isna()
    .mean()
)


family_missingness = []


for family, columns in families.items():

    values = (
        sample[
            columns
        ]
        .isna()
        .mean()
    )

    family_missingness.append(
        {
            "family":
                family,

            "feature_count":
                len(columns),

            "mean_missing_pct":
                values.mean() * 100,

            "median_missing_pct":
                values.median() * 100,

            "max_missing_pct":
                values.max() * 100,
        }
    )


family_missingness = (
    pd.DataFrame(
        family_missingness
    )
    .sort_values(
        "mean_missing_pct",
        ascending=False,
    )
)


family_missingness

,family,feature_count,mean_missing_pct,median_missing_pct,max_missing_pct
0,D,96,23.430975,3.2072,99.9412
3,B,40,8.740700,0.0106,99.3168
4,R,28,6.624214,0.0000,94.0204
1,S,21,5.574381,0.0000,53.1792
2,P,3,2.100267,0.8084,5.4924


In [9]:
# We scan only customer_ID and S_2 from the full CSV.
# This gives exact row count, exact date range and statements-per-customer
# without loading all 188 feature columns.

row_count = 0

minimum_date = None
maximum_date = None

customer_counts = {}


for chunk_number, chunk in enumerate(

    pd.read_csv(
        TRAIN_PATH,
        usecols=[
            "customer_ID",
            "S_2",
        ],
        chunksize=500_000,
    ),

    start=1,
):

    row_count += len(
        chunk
    )

    chunk[
        "S_2"
    ] = pd.to_datetime(
        chunk[
            "S_2"
        ],
        errors="coerce",
    )

    current_min = (
        chunk[
            "S_2"
        ].min()
    )

    current_max = (
        chunk[
            "S_2"
        ].max()
    )

    if (
        minimum_date is None
        or current_min < minimum_date
    ):
        minimum_date = (
            current_min
        )

    if (
        maximum_date is None
        or current_max > maximum_date
    ):
        maximum_date = (
            current_max
        )

    counts = (
        chunk[
            "customer_ID"
        ]
        .value_counts()
    )

    for customer, count in counts.items():

        customer_counts[
            customer
        ] = (
            customer_counts.get(
                customer,
                0,
            )
            + int(count)
        )

    print(
        f"Chunk {chunk_number} finished. "
        f"Rows counted: {row_count:,}"
    )

Chunk 1 finished. Rows counted: 500,000


Chunk 2 finished. Rows counted: 1,000,000


Chunk 3 finished. Rows counted: 1,500,000


Chunk 4 finished. Rows counted: 2,000,000


Chunk 5 finished. Rows counted: 2,500,000


...

Chunk 7 finished. Rows counted: 3,500,000


Chunk 8 finished. Rows counted: 4,000,000


Chunk 9 finished. Rows counted: 4,500,000


Chunk 10 finished. Rows counted: 5,000,000


Chunk 11 finished. Rows counted: 5,500,000
Chunk 12 finished. Rows counted: 5,531,451


In [10]:
statement_counts = pd.Series(
    customer_counts,
    name="n_statements",
)


print(
    "Full train_data row count:",
    f"{row_count:,}",
)

print(
    "Unique customers:",
    f"{len(statement_counts):,}",
)

print(
    "Statement date range:",
    minimum_date,
    "to",
    maximum_date,
)


print(
    statement_counts.describe()
)

Full train_data row count: 5,531,451
Unique customers: 458,913
Statement date range: 2017-03-01 00:00:00 to 2018-03-31 00:00:00


count    458913.000000
mean         12.053376
std           2.610273
min           1.000000
25%          13.000000
50%          13.000000
75%          13.000000
max          13.000000
Name: n_statements, dtype: float64


In [11]:
# This final table stores the major data-understanding facts
# so we can later turn actual results into polished Markdown.

data_summary = pd.DataFrame(
    [
        {
            "metric":
                "statement_rows",

            "value":
                row_count,
        },
        {
            "metric":
                "customers",

            "value":
                len(statement_counts),
        },
        {
            "metric":
                "train_data_columns",

            "value":
                len(sample.columns),
        },
        {
            "metric":
                "behavioral_features",

            "value":
                len(feature_columns),
        },
        {
            "metric":
                "default_rate",

            "value":
                labels[
                    "target"
                ].mean(),
        },
        {
            "metric":
                "first_statement",

            "value":
                minimum_date,
        },
        {
            "metric":
                "last_statement",

            "value":
                maximum_date,
        },
    ]
)


data_summary

,metric,value
0,statement_rows,5531451
1,customers,458913
2,train_data_columns,190
3,behavioral_features,188
4,default_rate,0.258934
5,first_statement,2017-03-01 00:00:00
6,last_statement,2018-03-31 00:00:00


## Summary

This notebook inspected the raw AMEX files only. It did not use PostgreSQL.

**What we did**
- Confirmed `train_data.csv` and `train_labels.csv` exist.
- Loaded the full label file for exact customer count and default rate.
- Read a 250,000-row statement sample for schema, grain, feature families, categoricals, and missingness.
- Scanned the full statement file using only `customer_ID` and `S_2` with chunksize, so the 15 GB file was never loaded with all columns.

**Results**
- Labels: **458,913** customers, **0** duplicate IDs, default rate **25.8934%**.
- Statements: **5,531,451** rows, **458,913** unique customers (matches the label file).
- Grain: one row = one customer × statement date. Most customers have **13** statements (mean 12.05, min 1, max 13).
- Columns: **190** total; **188** behavioral features after dropping `customer_ID` and `S_2`.
- Families: D 96, B 40, R 28, S 21, P 3. Known categoricals: 11.
- Date range: **2017-03-01 to 2018-03-31**.
- Sample missingness is highest in D features (mean 23.4%, max 99.9%) and lowest in P features (mean 2.1%).

These counts are the inputs used by the warehouse and later notebooks.